# Face Detection & Recognition Pipeline — Colab Demo

**GPU: T4 recommended.** Set it via *Runtime → Change runtime type → T4 GPU* before running.

---

## Overview

Code and sample test images are cloned from a **public GitHub repository** in Section 2.

> **SwinFace weights** (`SwinFace_MS1MV2.pth`, ~300 MB) cannot be stored on GitHub due to size.  
> Section 2b downloads them from Google Drive via `gdown` (commented out by default).  
> All other models — ArcFace, Facenet512, VGG-Face, InsightFace (buffalo_l) — are downloaded automatically on first use.

---

## Pipeline sections

| Section | Task | Input |
|---------|------|-------|
| 4 | **Detection** — locate faces and draw bounding boxes | images in `test_images/detection/` |
| 5 | **Verification** — determine if two face images belong to the same person | `img1.jpg` and `img2.jpg` |
| 6 | **Identification** — match a probe face against a gallery | one sub-folder per identity |

## 1 — Install dependencies

In [ ]:
!pip install deepface -q
!pip install retina-face -q
!pip install opencv-python -q
!pip install tensorflow -q
!pip install insightface onnxruntime-gpu -q
!pip install torchmetrics -q
!pip install timm -q
!pip install mtcnn -q
!pip install ultralytics -q
print("All packages installed.")

## 1b — Model cache (optional but recommended)

DeepFace and InsightFace download their weights on the **first run** (~500 MB total for ArcFace + buffalo_l). This is fine for a single session but takes a few minutes each time the runtime resets.

If you have the weights saved somewhere (e.g. your Drive from a previous session), set `MODELS_DIR` to that path and they will not be re-downloaded. Otherwise leave the default `/content/models` and they will download once into the Colab session.

In [ ]:
import os

# Change to a Drive path if you have cached weights, e.g.:
# "/content/drive/MyDrive/masters_thesis/face_detection_recognition/models"
MODELS_DIR = "/content/models"

os.makedirs(MODELS_DIR, exist_ok=True)
os.environ['DEEPFACE_HOME'] = MODELS_DIR   # inherited by all !python subprocesses
print(f"DEEPFACE_HOME = {MODELS_DIR}")
print("Models will be downloaded here on first use and reused in subsequent cells.")

## 2 — Clone repository

Clones the public GitHub repository and copies the scripts, source code, and sample test images into `/content`.

In [ ]:
import os, shutil, subprocess

# ── SET YOUR REPO URL ─────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/DominikTruchly/Face_Detection_Recognition_framework"
# ─────────────────────────────────────────────────────────────────────────────

WORKDIR  = "/content"
REPO_DIR = os.path.join(WORKDIR, "repo")

# ---- clone / refresh repo ----
if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print(f"Cloning {GITHUB_REPO} ...")
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, REPO_DIR], check=True)

# ---- copy code files into /content so imports resolve correctly ----
for fname in ["run_detection.py", "run_verification.py", "run_identification.py"]:
    s = os.path.join(REPO_DIR, fname)
    if os.path.exists(s):
        shutil.copy2(s, os.path.join(WORKDIR, fname))

src_local = os.path.join(WORKDIR, "src")
if os.path.exists(src_local):
    shutil.rmtree(src_local)
shutil.copytree(os.path.join(REPO_DIR, "src"), src_local)

# ---- copy sample test images from repo ----
for sub in ["detection", "verification", "identification"]:
    src = os.path.join(REPO_DIR, "test_images", sub)
    dst = os.path.join(WORKDIR, "test_images", sub)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        n = sum(len(f) for _, _, f in os.walk(dst))
        print(f"Copied test_images/{sub}/ ({n} files)")
    else:
        os.makedirs(dst, exist_ok=True)
        print(f"NOTE: test_images/{sub}/ not found in repo — created empty folder.")

os.makedirs(os.path.join(WORKDIR, "results"), exist_ok=True)
os.makedirs(os.path.join(WORKDIR, "weights"), exist_ok=True)
os.chdir(WORKDIR)
print(f"\nReady. Working directory: {os.getcwd()}")

## 2b — Download SwinFace weights (optional)

Only needed if you want to test the SwinFace model. Skip this section for ArcFace, Facenet512, VGG-Face, and InsightFace.

Weights are downloaded from Google Drive via `gdown`. The cell is **commented out** — uncomment to run.

In [ ]:
# Uncomment the lines below to download SwinFace weights from Google Drive.
# Re-running this cell is safe — gdown skips the download if the file already exists.

# import os
# !pip install gdown -q
#
# SWINFACE_FILE_ID = "1fi4IuuFV8NjnWm-CufdrhMKrkjxhSmjx"
# SWINFACE_OUT     = "/content/weights/SwinFace_MS1MV2.pth"
#
# if not os.path.exists(SWINFACE_OUT):
#     print("Downloading SwinFace weights (~300 MB)...")
#     !gdown {SWINFACE_FILE_ID} -O {SWINFACE_OUT}
#     print("Done.")
# else:
#     print("SwinFace weights already present, skipping download.")

## 3 — Inspect test images

Sample images are included in the repository and copied here in Section 2.

| Folder | Layout |
|--------|--------|
| `test_images/detection/` | Flat folder of face images |
| `test_images/verification/` | Face images — used for 1-vs-1 comparisons |
| `test_images/identification/` | One sub-folder per person (folder name = identity label) |

In [ ]:
import os
from IPython.display import display
from PIL import Image

def list_images(folder):
    exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    found = []
    for root, _, files in os.walk(folder):
        for f in sorted(files):
            if f.lower().endswith(exts):
                found.append(os.path.relpath(os.path.join(root, f), folder))
    return found

for task in ["detection", "verification", "identification"]:
    path = f"/content/test_images/{task}"
    imgs = list_images(path)
    print(f"\ntest_images/{task}/  ({len(imgs)} image(s))")
    for p in imgs:
        print(f"  {p}")

# Preview detection images (up to 4)
print("\n--- Detection preview ---")
det_imgs = list_images("/content/test_images/detection")[:4]
if det_imgs:
    for p in det_imgs:
        display(Image.open(f"/content/test_images/detection/{p}").resize((300, 300)))
else:
    print("No images yet — add some to test_images/detection/ via the file browser.")

## 4 — Face Detection

Detects all faces in every image under `test_images/detection/`.
Raw predictions are saved to `results/.../raw_predictions.csv`.
`--save_faces` also saves annotated images to `results/.../visualized_images/`.

**Detector options:** `retinaface` · `mtcnn` · `opencv` · `yolov8n` · `ssd` · `dlib` · `mediapipe` · `centerface`

In [ ]:
# ── Detection on all images in test_images/detection/ ──────────────────────
!python run_detection.py \
    --detector retinaface \
    --img_dir /content/test_images/detection \
    --output_dir /content/results/detection_demo \
    --save_faces

In [ ]:
# ── (Optional) single image ─────────────────────────────────────────────────
# !python run_detection.py \
#     --detector retinaface \
#     --img_dir /content/test_images/detection/my_photo.jpg

In [ ]:
import glob, os
from IPython.display import display
from PIL import Image

vis_dir = "/content/results/detection_demo/visualized_images"
images = sorted(glob.glob(os.path.join(vis_dir, "**", "*.*"), recursive=True))
images = [p for p in images if p.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

if images:
    print(f"Showing {min(4, len(images))} of {len(images)} visualized image(s):")
    for p in images[:4]:
        display(Image.open(p))
else:
    print(f"No visualized images found in {vis_dir}\nDid you pass --save_faces?")

## 5 — Face Verification (1-vs-1)

Compares two images and reports the cosine distance and a same-person verdict.  
The cells below show a **same-person** pair and a **different-person** pair using the provided sample images.

**Recognizer options:** `ArcFace` · `Facenet512` · `Facenet` · `VGG-Face` · `InsightFace` · `InsightFace_Custom` · `SwinFace` · `ViT_timm`  
**Detector options:** `retinaface` · `mtcnn` · `opencv`

In [ ]:
# ── Same-person pair (expected: low distance, verified = YES) ────────────────
!python run_verification.py \
    --recognizer ArcFace \
    --detector retinaface \
    --img1 /content/test_images/verification/Tom_Cruise_0001.jpg \
    --img2 /content/test_images/verification/Tom_Cruise_0002.jpg

In [ ]:
# ── Different-person pair (expected: high distance, verified = NO) ───────────
!python run_verification.py \
    --recognizer ArcFace \
    --detector retinaface \
    --img1 /content/test_images/verification/Tom_Cruise_0001.jpg \
    --img2 /content/test_images/verification/Wayne_Gretzky_0001.jpg

## 6 — Face Identification (gallery → probe matching)

Builds a gallery from `test_images/identification/` (one sub-folder per person).  
The first `--num_shots` images per identity become gallery references; the rest become probes.  
Reports **Rank-1 / Rank-5 / Rank-10** accuracy.

> Minimum requirement: at least **2 images per person** (1 gallery + 1 probe).

In [ ]:
# ── 1-shot identification (ArcFace + RetinaFace) ─────────────────────────────
!python run_identification.py \
    --dataset_path /content/test_images/identification \
    --recognizer ArcFace \
    --detector retinaface \
    --num_shots 1 \
    --results_base_dir /content/results

In [ ]:
# ── (Optional) InsightFace_Custom ────────────────────────────────────────────
# !python run_identification.py \
#     --dataset_path /content/test_images/identification \
#     --recognizer InsightFace_Custom \
#     --detector retinaface \
#     --num_shots 1 \
#     --results_base_dir /content/results

In [ ]:
# ── (Optional) SwinFace ──────────────────────────────────────────────────────
# !python run_identification.py \
#     --dataset_path /content/test_images/identification \
#     --recognizer SwinFace \
#     --detector retinaface \
#     --num_shots 1 \
#     --results_base_dir /content/results

## 7 — View results

In [ ]:
import json, glob, os

metric_files = glob.glob("/content/results/**/summary_metrics.json", recursive=True)
ident_files  = glob.glob("/content/results/**/identification_summary.json", recursive=True)

for f in sorted(metric_files + ident_files):
    print(f"\n{'='*60}")
    print(f"  {os.path.relpath(f, '/content/results')}")
    print('='*60)
    with open(f) as fh:
        print(json.dumps(json.load(fh), indent=2))